In [1]:
#أولًا: خوارزمية RC4 – توليد سلسلة المفتاح


#المرحلة 1: خوارزمية تهيئة المفتاح (KSA)
def rc4_ksa(key):
    key = [ord(c) for c in key]
    S = list(range(256))
    j = 0

    for i in range(256):
        j = (j + S[i] + key[i % len(key)]) % 256
        S[i], S[j] = S[j], S[i]

    return S


#المرحلة 2: توليد السلسلة المفتاحية (PRGA)
def rc4_prga(S, length):
    i = j = 0
    keystream = []

    for _ in range(length):
        i = (i + 1) % 256
        j = (j + S[i]) % 256
        S[i], S[j] = S[j], S[i]
        K = S[(S[i] + S[j]) % 256]
        keystream.append(K)

    return keystream


#توليد سلسلة المفتاح كاملة
def rc4_keystream(key, length):
    S = rc4_ksa(key)
    return rc4_prga(S, length)


In [2]:
#مثال
ks = rc4_keystream("KEY", 16)
print(ks)


[149, 96, 252, 84, 182, 143, 68, 248, 54, 132, 251, 134, 240, 95, 211, 45]


In [3]:
#ثانيًا: اختبار المشتق الثنائي (Binary Derivative Test)
def binary_derivative_test(keystream):
    bits = "".join(format(byte, '08b') for byte in keystream)

    derivative = ""
    for i in range(len(bits) - 1):
        derivative += str(int(bits[i]) ^ int(bits[i + 1]))

    ones = derivative.count('1')
    zeros = derivative.count('0')

    return {
        "length": len(derivative),
        "ones": ones,
        "zeros": zeros,
        "ratio_ones": ones / len(derivative)
    }


In [4]:
#ثالثًا: اختبار نقطة التغير (Change Point Test)
def change_point_test(keystream):
    n = len(keystream)
    mean_total = sum(keystream) / n

    max_diff = 0
    change_point = 0

    for k in range(1, n):
        mean1 = sum(keystream[:k]) / k
        mean2 = sum(keystream[k:]) / (n - k)
        diff = abs(mean1 - mean2)

        if diff > max_diff:
            max_diff = diff
            change_point = k

    return {
        "change_point": change_point,
        "max_difference": max_diff,
        "global_mean": mean_total
    }


In [5]:
#مثال تشغيل كامل
keystream = rc4_keystream("SECRET", 100)

print("Binary Derivative Test:")
print(binary_derivative_test(keystream))

print("\nChange Point Test:")
print(change_point_test(keystream))


Binary Derivative Test:
{'length': 799, 'ones': 397, 'zeros': 402, 'ratio_ones': 0.49687108886107634}

Change Point Test:
{'change_point': 98, 'max_difference': 72.90816326530611, 'global_mean': 127.45}
